# Inferring

Infer sentiment, emotions, entities and topics from product reviews and news
articles — that is, ask a model to read a text and return a **label** rather
than prose.

## 0. What is in this notebook, and what it found

**Part 1** is the guided lab, preserved as shipped. **Part 2** is the exercise:
three prompt versions over six reviews chosen to break single-label sentiment.

| | What changed | What it was testing |
|---|---|---|
| **V1** | the lab's own combined prompt | does free-form "answer as JSON" actually parse? |
| **V2** | a closed label set and an explicit `unknown` rule | does offering `mixed` fix the mixed review? |
| **V3** | one sentiment per aspect, not one per review | can it separate a good product from bad service? |

Two changes were needed to the guided cells, both flagged where they occur:

1. **`gpt-3.5-turbo` is retired** and now returns 404, so a current model is
   named instead. Nothing else about the guided prompts is altered.
2. **The lab's final cell raises `ValueError` as shipped.** Section 7 keeps the
   original line, shows the exception, and explains why string-splitting a JSON
   object was always going to break.

*Note on "ground truth": the expected labels in Part 2 are **my** readings of
texts I wrote. They are a fixed yardstick, not objective fact, and the places
where the model disagrees with me are discussed rather than scored as simple
errors.*

---

# Part 1 — the guided lab

Everything from here to the Exercise heading is the lab as provided.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Presence only - the key itself is never printed.
print("OPENAI_API_KEY found:", bool(OPENAI_API_KEY))

OPENAI_API_KEY found: True


🔧 **Change 1 of 2.** The lab calls `gpt-3.5-turbo`. That model has been
retired and the API now answers:

```
NotFoundError: Error code: 404 - The model `gpt-3.5-turbo` has been deprecated
```

So `MODEL` below names a current one. The function is otherwise the lab's,
`temperature=0` included.

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def get_completion(prompt, model=None):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

print("Model:", MODEL)

Model: gpt-4o-mini


## Product review text

In [3]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)

In [4]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the product review is positive. The reviewer expresses satisfaction with the lamp's features, the quick delivery, and the responsive customer service. They highlight the company's care for its customers, which further reinforces the positive sentiment.


In [5]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

positive


## Identify types of emotions

In [6]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

satisfaction, gratitude, relief, appreciation, trust


## Identify anger

In [7]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews

In [8]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

```json
{"Item":"lamp","Brand":"Lumina"}
```


## Doing multiple tasks at once

In [9]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

```json
{
  "Sentiment": "positive",
  "Anger": false,
  "Item": "lamp",
  "Brand": "Lumina"
}
```


## Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.

In [10]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

Five topics discussed in the article are requested from the model in a format that each item is one or two words long and in a comma-separated list. ChatGPT returns the topics as government surveys, job satisfaction, NASA, etc.

In [11]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas without numbering them.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

Employee Satisfaction, NASA, Survey Results, Job Improvement, Government Response


In [12]:
response.split(sep=', ')

['Employee Satisfaction',
 'NASA',
 'Survey Results',
 'Job Improvement',
 'Government Response']

## Make a news alert for certain topics

The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:

In [13]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

In [14]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as a dictionay where the key is a topic and the value is 0 or 1 for each topic if it appears.\

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

```json
{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 0
}
```


In [15]:
for i in response.split(', '):
    print(i)

```json
{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 0
}
```


🔧 **Change 2 of 2 — the lab's last cell crashes.**

The shipped line is:

```python
topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep=', ')}
```

It assumes `response` is a flat `key: value, key: value` string. It is not —
the model returns a pretty-printed JSON object across several lines, as the
output two cells above shows. So `response.split(', ')` returns the whole blob
as one item, `i.split(': ')[1]` is `'1,\n    "local government"'`, and `int()`
raises.

The next cell runs the original line and catches the exception, so the failure
is visible rather than described.

In [16]:
# The lab's own line, kept exactly, with the exception caught.
try:
    topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep=', ')}
    print("parsed:", topic_dict)
except (ValueError, IndexError) as exc:
    print("the lab's parse raises %s: %s" % (type(exc).__name__, exc))
    print()
    print("what it was handed:")
    print(repr(response[:120]), "...")

the lab's parse raises ValueError: invalid literal for int() with base 10: '1,\n    "local government"'

what it was handed:
'```json\n{\n    "nasa": 1,\n    "local government": 0,\n    "engineering": 0,\n    "employee satisfaction": 1,\n    "federal g' ...


The fix is not a cleverer split. It is to stop treating structured output as
text: the model was asked for JSON, so parse it as JSON.

In [17]:
import json, re

def parse_json_reply(text):
    """Parse a model reply that should be JSON.

    Models commonly wrap JSON in a ```json fence even when not asked to, so the
    fence is stripped first. Returns (value, error) - never raises, because a
    parse failure is a result worth measuring, not a crash.
    """
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as exc:
        return None, str(exc)


topic_dict, error = parse_json_reply(response)
print("parsed:", topic_dict)
print("error :", error)

if topic_dict and topic_dict.get("nasa") == 1:
    print("\nALERT: New NASA story!")

parsed: {'nasa': 1, 'local government': 0, 'engineering': 0, 'employee satisfaction': 1, 'federal government': 0}
error : None

ALERT: New NASA story!


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

---

# Part 2 — the exercise

Everything below is mine. Part 1 above is the lab as given.

The guided lab infers from **one** review, and that review is unambiguous: a
delighted customer, one product, one named brand. Every prompt in it succeeds.
That is a demonstration, not a test — nothing in it could have failed.

So Part 2 keeps the task identical and changes the **inputs**: six reviews
chosen so that single-label sentiment has somewhere to go wrong.

## 1. Six reviews, and what each one is for

| id | What it probes |
|---|---|
| `lamp_positive` | the lab's own review, kept as a control |
| `mixed_product_vs_service` | a **good product** and **bad service** in one text |
| `sarcastic_negative` | praise words, negative meaning |
| `hedged_negation` | "not that it's bad… not what you'd call good either" |
| `angry_negative` | negative **and** angry, to separate the two questions |
| `no_brand_named` | no brand in the text — will it invent one? |

The `expected` labels are my readings. They are stated up front so they cannot
be adjusted after seeing what the model said.

In [18]:
REVIEWS = [
    {
        "id": "lamp_positive",
        "probes": "control - the lab's own review",
        "text": lamp_review,
        "expected": {"sentiment": "positive", "anger": False,
                     "item": "lamp", "brand": "Lumina"},
    },
    {
        "id": "mixed_product_vs_service",
        "probes": "good product, bad service, in one review",
        "text": """The blender itself is superb - it crushes ice in seconds and the jug
feels like it will outlive me. Getting hold of it was another story. The courier
lost the first one, the replacement took two weeks, and nobody at Vitamix
answered three emails. Great machine, appalling service.""",
        "expected": {"sentiment": "mixed", "anger": False,
                     "item": "blender", "brand": "Vitamix"},
    },
    {
        "id": "sarcastic_negative",
        "probes": "positive words, negative meaning",
        "text": """Fantastic. Truly. The chair arrived with three legs, which I am told is
the new minimalist look. The assembly manual is a masterpiece of abstract art.
Ten out of ten, would wobble again.""",
        "expected": {"sentiment": "negative", "anger": False,
                     "item": "chair", "brand": "unknown"},
    },
    {
        "id": "hedged_negation",
        "probes": "double negation and hedging, no strong words either way",
        "text": """I would not say I was impressed. It is not that the headphones are bad,
exactly - they are just not what you would call good either. I have had worse.
I have also had considerably better for the money.""",
        "expected": {"sentiment": "negative", "anger": False,
                     "item": "headphones", "brand": "unknown"},
    },
    {
        "id": "angry_negative",
        "probes": "negative AND angry - the lab asks these as separate questions",
        "text": """This is the third time I am writing about this. The Kenwood mixer died
inside the warranty period, your support line hung up on me twice, and now you
are telling me the receipt I sent does not count. It is a disgrace and I want
my money back.""",
        "expected": {"sentiment": "negative", "anger": True,
                     "item": "mixer", "brand": "Kenwood"},
    },
    {
        "id": "no_brand_named",
        "probes": "no brand in the text - the prompt says to answer 'unknown'",
        "text": """Arrived on time and does what it says. The stand is sturdy enough for a
27-inch monitor and the cable routing is tidy. Nothing else to report.""",
        "expected": {"sentiment": "positive", "anger": False,
                     "item": "monitor stand", "brand": "unknown"},
    },
]

import pandas as pd
display(pd.DataFrame([{"id": r["id"], "probes": r["probes"],
                       "expected sentiment": r["expected"]["sentiment"],
                       "expected brand": r["expected"]["brand"]}
                      for r in REVIEWS]))

,id,probes,expected sentiment,expected brand
0,lamp_positive,control - the lab's own review,positive,Lumina
1,mixed_product_vs_service,"good product, bad service, in one review",mixed,Vitamix
2,sarcastic_negative,"positive words, negative meaning",negative,unknown
3,hedged_negation,"double negation and hedging, no strong words e...",negative,unknown
4,angry_negative,negative AND angry - the lab asks these as sep...,negative,Kenwood
5,no_brand_named,no brand in the text - the prompt says to answ...,positive,unknown


## 2. Three prompt versions

Each takes one review and returns the same four fields, so they are directly
comparable. Only the instruction changes.

**V1 — the lab's own combined prompt.** Cell 15 of Part 1, unchanged in
substance. It asks for JSON in prose, with no stated label set.
*Hypothesis: it will work on easy reviews and flatten the mixed one.*

**V2 — a closed label set.** The same four fields, but `sentiment` must be one
of four named values, `mixed` among them, and the `unknown` rule for absent
information is spelled out.
*Hypothesis: giving it `mixed` is enough to fix the mixed review.*

**V3 — one sentiment per aspect.** Instead of a single verdict, a sentiment for
the product, the delivery and the support separately, plus an overall.
*Hypothesis: the mixed review is not a labelling problem but a modelling one —
one field cannot hold two opinions.*

In [19]:
def v1_prompt(review):
    """The lab's combined prompt from Part 1, applied to a new review."""
    return f"""
Identify the following items from the review text:
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{review}'''
"""


def v2_prompt(review):
    """Same four fields, but the label set is closed and 'unknown' is defined."""
    return f"""From the review below, identify four things.

sentiment  one of exactly: positive, negative, neutral, mixed.
           Use "mixed" when the reviewer is clearly positive about some part
           and clearly negative about another. Do not average them away.
           Judge the writer's meaning, not their vocabulary - praise used
           sarcastically is negative.
anger      true only if the writer is angry, not merely dissatisfied.
item       the thing purchased, in one or two words.
brand      the company that made it. If the review does not name a company,
           answer exactly "unknown". Do not infer it from the product type.

Return only a JSON object with the keys sentiment, anger, item, brand.

Review: '''{review}'''
"""


def v3_prompt(review):
    """One sentiment per aspect - the review is allowed to hold two opinions."""
    return f"""Read the review below and rate it aspect by aspect.

For each of product, delivery and support, give one of:
  positive, negative, not_mentioned

Then give an overall sentiment, one of:
  positive, negative, neutral, mixed
Use "mixed" when the aspects disagree with each other.

Also give:
  anger  true only if the writer is angry, not merely dissatisfied
  item   the thing purchased, one or two words
  brand  the company that made it, or exactly "unknown" if the review does
         not name one

Judge meaning rather than vocabulary; sarcastic praise is negative.

Return only a JSON object with the keys:
product, delivery, support, overall, anger, item, brand

Review: '''{review}'''
"""


VERSIONS = {"V1 lab prompt": v1_prompt,
            "V2 closed label set": v2_prompt,
            "V3 per aspect": v3_prompt}

print("prompt lengths on the control review:")
for name, builder in VERSIONS.items():
    print("  %-22s %4d chars" % (name, len(builder(REVIEWS[0]["text"]))))

prompt lengths on the control review:
  V1 lab prompt           967 chars
  V2 closed label set    1198 chars
  V3 per aspect          1148 chars


## 3. Run all three over all six reviews

18 calls at `temperature=0`. Replies are parsed with `parse_json_reply` from
Part 1, which records a parse failure instead of raising — whether the output
parses at all is one of the things being measured.

In [20]:
import time

def normalise(parsed):
    """V1 capitalises its keys, V2 and V3 do not. Compare like with like."""
    if not isinstance(parsed, dict):
        return {}
    lower = {str(k).lower(): v for k, v in parsed.items()}
    out = {
        "sentiment": lower.get("sentiment") or lower.get("overall"),
        "anger": lower.get("anger"),
        "item": lower.get("item"),
        "brand": lower.get("brand"),
    }
    for aspect in ("product", "delivery", "support"):
        if aspect in lower:
            out[aspect] = lower[aspect]
    return {k: (v.lower() if isinstance(v, str) else v) for k, v in out.items()}


runs = []
started = time.perf_counter()
for review in REVIEWS:
    for name, builder in VERSIONS.items():
        raw = get_completion(builder(review["text"]))
        parsed, error = parse_json_reply(raw)
        runs.append({"review": review["id"], "version": name, "raw": raw,
                     "parsed": parsed, "parse_error": error,
                     "fields": normalise(parsed),
                     "expected": review["expected"]})
        print("%-26s %-22s %s" % (review["id"], name,
                                  "parsed" if parsed is not None else "PARSE FAILED"))

print("\n%d calls in %.0fs" % (len(runs), time.perf_counter() - started))

lamp_positive              V1 lab prompt          parsed
lamp_positive              V2 closed label set    parsed
lamp_positive              V3 per aspect          parsed
mixed_product_vs_service   V1 lab prompt          parsed
mixed_product_vs_service   V2 closed label set    parsed
mixed_product_vs_service   V3 per aspect          parsed
sarcastic_negative         V1 lab prompt          parsed
sarcastic_negative         V2 closed label set    parsed
sarcastic_negative         V3 per aspect          parsed
hedged_negation            V1 lab prompt          parsed
hedged_negation            V2 closed label set    parsed
hedged_negation            V3 per aspect          parsed
angry_negative             V1 lab prompt          parsed
angry_negative             V2 closed label set    parsed
angry_negative             V3 per aspect          parsed
no_brand_named             V1 lab prompt          parsed
no_brand_named             V2 closed label set    parsed
no_brand_named             V3 p

## 4. Did it parse, and did it agree with me?

Two separate questions, deliberately kept apart. A reply can parse perfectly and
still be wrong, which is the same distinction the previous lab ran into with an
order slip that added up but charged the wrong price.

In [21]:
def agrees(got, want, field):
    """Field-level agreement, tolerant about wording but not about meaning."""
    g, w = got.get(field), want.get(field)
    if field == "anger":
        return bool(g) == bool(w)
    if g is None:
        return False
    g, w = str(g).lower().strip(), str(w).lower().strip()
    if field in ("item", "brand"):
        return g == w or w in g or g in w      # "monitor stand" vs "stand"
    return g == w


rows = []
for r in runs:
    row = {"review": r["review"], "version": r["version"],
           "parsed": r["parsed"] is not None}
    for field in ("sentiment", "anger", "item", "brand"):
        row[field] = agrees(r["fields"], r["expected"], field)
    row["got sentiment"] = r["fields"].get("sentiment")
    row["want sentiment"] = r["expected"]["sentiment"]
    rows.append(row)

detail = pd.DataFrame(rows)
display(detail)

summary = (detail.groupby("version")[["parsed", "sentiment", "anger", "item", "brand"]]
                 .sum().astype(int))
summary.insert(0, "of", len(REVIEWS))
print("\nagreement with my labels, out of %d reviews:" % len(REVIEWS))
display(summary)

,review,version,parsed,sentiment,anger,item,brand,got sentiment,want sentiment
0,lamp_positive,V1 lab prompt,True,True,True,True,True,positive,positive
1,lamp_positive,V2 closed label set,True,True,True,True,True,positive,positive
2,lamp_positive,V3 per aspect,True,True,True,True,True,positive,positive
3,mixed_product_vs_service,V1 lab prompt,True,False,False,True,True,negative,mixed
4,mixed_product_vs_service,V2 closed label set,True,True,False,True,True,mixed,mixed
5,mixed_product_vs_service,V3 per aspect,True,True,True,True,True,mixed,mixed
6,sarcastic_negative,V1 lab prompt,True,False,True,True,True,positive,negative
7,sarcastic_negative,V2 closed label set,True,False,True,True,True,mixed,negative
8,sarcastic_negative,V3 per aspect,True,True,True,True,True,negative,negative
9,hedged_negation,V1 lab prompt,True,True,True,True,True,negative,negative



agreement with my labels, out of 6 reviews:


,of,parsed,sentiment,anger,item,brand
version,,,,,,
V1 lab prompt,6,6,4,5,6,6
V2 closed label set,6,6,4,5,6,6
V3 per aspect,6,6,5,6,6,6


In [22]:
# Where the versions disagree with me, and with each other.
print("Sentiment, review by review")
print("=" * 78)
pivot = detail.pivot(index="review", columns="version", values="got sentiment")
pivot["expected"] = [next(r["expected"]["sentiment"] for r in REVIEWS if r["id"] == i)
                     for i in pivot.index]
display(pivot)

print()
for r in runs:
    if not agrees(r["fields"], r["expected"], "sentiment"):
        print("%-26s %-22s said %-10s I said %s"
              % (r["review"], r["version"], r["fields"].get("sentiment"),
                 r["expected"]["sentiment"]))

Sentiment, review by review


version,V1 lab prompt,V2 closed label set,V3 per aspect,expected
review,,,,
angry_negative,negative,negative,negative,negative
hedged_negation,negative,mixed,mixed,negative
lamp_positive,positive,positive,positive,positive
mixed_product_vs_service,negative,mixed,mixed,mixed
no_brand_named,positive,positive,positive,positive
sarcastic_negative,positive,mixed,negative,negative



mixed_product_vs_service   V1 lab prompt          said negative   I said mixed
sarcastic_negative         V1 lab prompt          said positive   I said negative
sarcastic_negative         V2 closed label set    said mixed      I said negative
hedged_negation            V2 closed label set    said mixed      I said negative
hedged_negation            V3 per aspect          said mixed      I said negative


In [23]:
# The brand question on its own: the prompt says answer "unknown" when absent.
# Inventing a brand is a hallucination with a clear right answer, so it is the
# cleanest single measurement in this notebook.
print("Reviews that name no brand - correct answer is 'unknown'")
print("=" * 78)
for r in runs:
    if r["expected"]["brand"] == "unknown":
        got = r["fields"].get("brand")
        verdict = "ok" if str(got).lower() == "unknown" else "INVENTED"
        print("%-26s %-22s -> %-20s %s" % (r["review"], r["version"], got, verdict))

Reviews that name no brand - correct answer is 'unknown'
sarcastic_negative         V1 lab prompt          -> unknown              ok
sarcastic_negative         V2 closed label set    -> unknown              ok
sarcastic_negative         V3 per aspect          -> unknown              ok
hedged_negation            V1 lab prompt          -> unknown              ok
hedged_negation            V2 closed label set    -> unknown              ok
hedged_negation            V3 per aspect          -> unknown              ok
no_brand_named             V1 lab prompt          -> unknown              ok
no_brand_named             V2 closed label set    -> unknown              ok
no_brand_named             V3 per aspect          -> unknown              ok


## 5. The aspect question

V3 is the only version that can report two opinions about one review. This cell
asks whether that extra structure actually bought anything, or whether V2's
`mixed` label was enough on its own.

In [24]:
print("V3's aspect breakdown")
print("=" * 78)
aspect_rows = []
for r in runs:
    if r["version"] != "V3 per aspect" or not r["parsed"]:
        continue
    f = r["fields"]
    aspect_rows.append({"review": r["review"],
                        "product": f.get("product"), "delivery": f.get("delivery"),
                        "support": f.get("support"), "overall": f.get("sentiment")})
aspects = pd.DataFrame(aspect_rows)
display(aspects)

def conflicted(row):
    seen = {row[a] for a in ("product", "delivery", "support")
            if row[a] not in (None, "not_mentioned")}
    return {"positive", "negative"} <= seen

if not aspects.empty:
    aspects["aspects disagree"] = aspects.apply(conflicted, axis=1)
    print("\nReviews where V3 found both a positive and a negative aspect:")
    print(list(aspects.loc[aspects["aspects disagree"], "review"]) or "none")

V3's aspect breakdown


,review,product,delivery,support,overall
0,lamp_positive,positive,positive,positive,positive
1,mixed_product_vs_service,positive,negative,negative,mixed
2,sarcastic_negative,negative,not_mentioned,not_mentioned,negative
3,hedged_negation,negative,not_mentioned,not_mentioned,mixed
4,angry_negative,negative,not_mentioned,negative,negative
5,no_brand_named,positive,positive,not_mentioned,positive



Reviews where V3 found both a positive and a negative aspect:
['mixed_product_vs_service']


## 6. Topic inference, with a trap

Part 1's news alert used a topic list where the absent topics were obviously
absent. This repeats it with topics that are **plausibly** related to a NASA
story without being discussed in it — engineering, budget, space exploration,
rocket launches. A model that pattern-matches on "NASA" rather than reading will
mark them present.

The fixed parse from section 7 of Part 1 is used, so a malformed reply is
recorded rather than crashing the cell.

In [25]:
TRAP_TOPICS = [
    "employee satisfaction",   # genuinely present
    "nasa",                    # genuinely present
    "federal government",      # genuinely present
    "engineering",             # NASA-adjacent, not discussed
    "rocket launches",         # NASA-adjacent, not discussed
    "budget cuts",             # government-adjacent, not discussed
    "space exploration",       # NASA-adjacent, not discussed
    "local government",        # the lab's own absent topic
]
PRESENT = {"employee satisfaction", "nasa", "federal government"}

topic_prompt = f"""
Determine whether each item in the following list of topics is discussed in the
text below, which is delimited with triple backticks.

A topic counts as present only if the text actually discusses it. Do not mark a
topic present merely because it is related to something the text mentions.

Answer with a JSON object where each key is a topic and each value is 0 or 1.

List of topics: {", ".join(TRAP_TOPICS)}

Text sample: '''{story}'''
"""

raw_topics = get_completion(topic_prompt)
topics, topic_error = parse_json_reply(raw_topics)
print("parse error:", topic_error)
print()

if topics:
    trap_rows = [{"topic": t, "model says": topics.get(t),
                  "actually present": int(t in PRESENT),
                  "correct": topics.get(t) == int(t in PRESENT)}
                 for t in TRAP_TOPICS]
    display(pd.DataFrame(trap_rows))
    false_positives = [t for t in TRAP_TOPICS
                       if topics.get(t) == 1 and t not in PRESENT]
    false_negatives = [t for t in TRAP_TOPICS
                       if topics.get(t) == 0 and t in PRESENT]
    print("falsely marked present:", false_positives or "none")
    print("missed:", false_negatives or "none")

parse error: None



,topic,model says,actually present,correct
0,employee satisfaction,1,1,True
1,nasa,1,1,True
2,federal government,1,1,True
3,engineering,0,0,True
4,rocket launches,0,0,True
5,budget cuts,0,0,True
6,space exploration,0,0,True
7,local government,0,0,True


falsely marked present: none
missed: none


## 7. Report

*`gpt-4o-mini`, `temperature=0`, six reviews x three prompt versions = 18 calls,
plus the topic trap. One run, 12 November 2026.*

### Scoreboard

| Version | parses | sentiment | anger | item | brand |
|---|---:|---:|---:|---:|---:|
| V1 lab prompt | 6/6 | **4/6** | 5/6 | 6/6 | 6/6 |
| V2 closed label set | 6/6 | **4/6** | 5/6 | 6/6 | 6/6 |
| V3 per aspect | 6/6 | **5/6** | 6/6 | 6/6 | 6/6 |

Everything except sentiment was saturated. Sentiment is where the whole lab is.

### Three of my four hypotheses were wrong

I wrote the hypotheses before running anything, so the failures are worth more
than the successes:

| Predicted | Actually |
|---|---|
| V1's loose "answer as JSON" would sometimes fail to parse | **Wrong.** 18/18 parsed. |
| The model would invent a brand where none was named | **Wrong.** 9/9 answered `unknown`. |
| It would mark NASA-adjacent topics present | **Wrong.** 8/8 correct, zero false positives. |
| Adding `mixed` would fix the mixed review | **Right**, but it broke a different one. |

The invented-brand test is the one I was most confident about, and it came back
completely clean. Three versions x three brand-less reviews, every one answered
`unknown`. On this evidence a clear "if it isn't there, say unknown" instruction
is simply obeyed.

### Where it actually went wrong

**V1 read a sarcastic review as positive.** Given a chair that arrived with
three legs, an assembly manual described as "a masterpiece of abstract art" and
the sign-off "ten out of ten, would wobble again", V1 answered **positive**.
This is the single clearest failure in the notebook, and it is the one the
guided lab could never have found, because the guided lab infers from one
genuinely delighted customer. Every prompt in Part 1 succeeds; none of them
could have failed.

**Telling the model about sarcasm was not enough.** V2's prompt says, in
as many words, "judge the writer's meaning, not their vocabulary - praise used
sarcastically is negative". V2 still got it wrong, answering `mixed`. Only V3
got it right - and V3's sarcasm instruction is identical. What changed was the
*structure*: V3 had to commit to a verdict on the product before giving an
overall, it rated the product `negative`, and the overall followed. The fix
came from making the model decompose the judgement, not from describing the
trap to it.

**Offering `mixed` gave it somewhere to hide.** V1 can only answer positive or
negative, and on the hedged review - "not that they are bad, exactly... not
what you would call good either" - that constraint forced a choice and V1 chose
correctly. V2 and V3, both of which can answer `mixed`, both did. So the extra
label fixed `mixed_product_vs_service` and broke `hedged_negation`. V3 still
nets ahead at 5/6, but adding a category is not a free win: it buys accuracy on
genuinely two-sided text and spends it on text the model is merely unsure about.

*Honesty about that case:* my label for `hedged_negation` is contestable.
"I have had worse, I have also had considerably better for the money" reads as
mildly negative to me, but `neutral` would be defensible. The sarcasm case is
not contestable in the same way - a three-legged chair is not a positive review.

**V1 and V2 called the mixed review angry; V3 did not.** "Appalling service" and
three unanswered emails are strong words, but the writer is dissatisfied rather
than angry - and that distinction matters, because the lab's own framing treats
sentiment and anger as separate questions precisely so that an angry customer
can be routed differently. V3 was the only version to keep them apart on all
six reviews.

### The bug the lab ships with

Part 1's final cell raises:

```
ValueError: invalid literal for int() with base 10: '1,\n    "local government"'
```

There are two reasons, not one. The model returns pretty-printed JSON spanning
several lines, so splitting on `', '` never produces `key: value` pairs. And the
reply arrives wrapped in a ` ```json ` fence that nothing in the lab strips.
The fix is not a better split - it is to stop treating structured output as
text and call `json.loads` on it, which is what the model was asked to produce
in the first place.

The fence is worth dwelling on. Nothing in the prompt asked for it; the model
added it unprompted. Any parser that assumes the reply begins with `{` will
break on output that is otherwise perfectly correct.

### One loose end

The guided prompt and my stricter trap prompt disagree about **federal
government** on the same story: the guided phrasing answered `0`, the trap
phrasing answered `1`. The story is a survey of federal agencies, so I had
labelled it present, and the stricter prompt agreed with me. Two phrasings of
the same question, one text, opposite answers. I am not going to claim which
phrasing is better from a single pair of calls - but it is a reminder that
topic detection is as sensitive to wording as sentiment is, and the guided
lab's `topic_dict['nasa'] == 1` alert would have behaved differently under a
prompt nobody would have thought to compare.

### What I learned

1. **A demonstration is not a test.** Part 1 infers from one unambiguous review
   and every prompt in it works. Nothing in it could have failed, so nothing in
   it told me anything about whether the prompts were good. The lab only became
   informative when the inputs got harder, and that cost nothing but writing
   six reviews.

2. **Structure beat instruction.** V2 and V3 carry the same sentence about
   sarcasm. V2 got the sarcastic review wrong and V3 got it right, because V3
   had to rate the product before summarising. Forcing the model to decompose a
   judgement worked where describing the failure mode to it did not.

3. **Every added label is a new place to be wrong.** `mixed` is the right answer
   for a review that praises the product and damns the delivery. It is the wrong
   answer for a review the model simply cannot read confidently, and the model
   cannot tell those two situations apart.

4. **Parse what you asked for.** The lab asked for JSON and then took the reply
   apart with `str.split`. That works until the model pretty-prints, or adds a
   code fence, or reorders keys - all of which are normal, none of which are
   errors on the model's part.

5. **Write down the prediction first.** Three of my four hypotheses were wrong,
   and I would not have noticed if I had not committed to them in writing before
   running anything. The brand-hallucination test I was most sure about returned
   9/9 clean, and the genuine failure - sarcasm - was one I had expected V2 to
   handle.

6. **Six reviews and one run is not an evaluation.** At `temperature=0` the API
   is still not bit-reproducible, the labels are mine rather than anyone's ground
   truth, and a single sarcastic review cannot establish that any version
   "handles sarcasm". These are observations about these texts, and the design
   is reusable; the numbers are not conclusions.